# Capítulo 2 — Playwright: Fundamentos para Pruebas E2E

**Prerequisito:** Haber completado el Capítulo 1 y tener la aplicación Flask corriendo.

**Objetivo:** Dominar los conceptos fundamentales de Playwright: instalación, arquitectura, tipos de locators, acciones disponibles, sistema de aserciones `expect()`, y las capacidades de diagnóstico (screenshots, trazas).

---

## 1. ¿Qué es Playwright?

**Playwright** es una librería de automatización de navegadores desarrollada por Microsoft (2020). Permite controlar Chromium, Firefox y WebKit (Safari) desde Python, JavaScript, Java o .NET usando una API unificada.

### Características que lo diferencian

| Característica | Descripción |
|---------------|-------------|
| **Auto-esperas** | Playwright espera automáticamente a que los elementos sean visibles e interactuables antes de actuar. Elimina la necesidad de `time.sleep()`. |
| **Aislamiento de contexto** | Cada contexto de navegador es completamente independiente (cookies, localStorage, caché). Ideal para tests paralelos. |
| **Intercepción de red** | Puede interceptar, modificar o bloquear peticiones HTTP durante los tests. |
| **Trazas** | Graba una traza completa (video, screenshots, logs de red) para diagnosticar fallos. |
| **Codegen** | Genera código de pruebas grabando las acciones del usuario en el navegador. |
| **Multi-browser** | Chromium, Firefox y WebKit desde la misma API. |

---

## 2. Instalación y configuración

```bash
pip install playwright pytest-playwright
playwright install chromium   # instala el binario del browser
```

### Dos modos de uso

**Modo síncrono** (usado en este taller — más legible para principiantes):
```python
from playwright.sync_api import sync_playwright
with sync_playwright() as pw:
    browser = pw.chromium.launch()
    page = browser.new_page()
    page.goto("https://ejemplo.com")
```

**Modo asíncrono** (para aplicaciones con `asyncio`):
```python
from playwright.async_api import async_playwright
async with async_playwright() as pw:
    ...
```

---

## 3. Arquitectura: Page, Browser, Context

Playwright organiza el control del navegador en tres niveles:

```
Browser
  └── BrowserContext  (aislamiento: cookies, localStorage, etc.)
        └── Page      (una pestaña del navegador)
              └── Frame  (iframes dentro de la página)
```

| Objeto | Descripción | Scope típico en tests |
|--------|-------------|----------------------|
| `Browser` | Instancia del navegador | Sesión completa |
| `BrowserContext` | Perfil aislado (como ventana de incógnito) | Por test o por clase |
| `Page` | Una pestaña | Por test |

La recomendación de Playwright es crear un `BrowserContext` nuevo por test para garantizar aislamiento total, aunque en este taller usamos uno por sesión para mayor velocidad.

---

## 4. Locators — Encontrando elementos en la página

Un **locator** es la forma de decirle a Playwright qué elemento HTML queremos interactuar. La elección del tipo de locator afecta directamente la **robustez** y **mantenibilidad** de los tests.

### 4.1 Jerarquía de robustez de locators

```
Más robusto  ← data-testid  → Explícito, no cambia con refactors de UI
             ← ARIA roles   → Semántico, accesible
             ← Texto exacto → Legible, puede cambiar con i18n
             ← CSS class    → Frágil si se refactoriza CSS
Más frágil   ← XPath        → Muy frágil, acoplado a la estructura HTML
```

### 4.2 Tipos de locators en Playwright

In [ ]:
# ──────────────────────────────────────────────────────────────
# DEMOSTRACIÓN DE TIPOS DE LOCATORS
# Ejecuta la app Flask antes de correr esta celda:
#   cd .. && python src/app.py
# ──────────────────────────────────────────────────────────────
from playwright.sync_api import sync_playwright

BASE_URL = "http://localhost:5000"

with sync_playwright() as pw:
    browser = pw.chromium.launch(headless=True)
    page = browser.new_page()
    page.goto(BASE_URL)

    print("=== Tipos de Locators ===\n")

    # 1. Por data-testid (RECOMENDADO — más robusto)
    titulo = page.locator("[data-testid='page-title']")
    print(f"[data-testid]  Título: '{titulo.inner_text()}'")

    # 2. Por role ARIA (semántico — muy recomendado)
    btn_agregar = page.get_by_role("button", name="Agregar")
    print(f"[role]         Botón 'Agregar' visible: {btn_agregar.is_visible()}")

    # 3. Por texto exacto
    h1 = page.get_by_text("Gestor de Tareas", exact=True)
    print(f"[texto]        H1 encontrado: {h1.count() > 0}")

    # 4. Por selector CSS
    form = page.locator("form.form-nueva")
    print(f"[CSS]          Formulario encontrado: {form.count() > 0}")

    # 5. Por placeholder del input
    input_titulo = page.get_by_placeholder("Escribe una nueva tarea...")
    print(f"[placeholder]  Input encontrado: {input_titulo.count() > 0}")

    # 6. XPath (desaconsejado — solo para casos sin alternativa)
    h1_xpath = page.locator("//h1")
    print(f"[XPath]        H1 por XPath: '{h1_xpath.inner_text()}'")

    print()
    print("Consejo: prioriza data-testid > roles ARIA > texto > CSS > XPath")

    browser.close()

---

## 5. Acciones — Interactuando con la página

Playwright ofrece un conjunto rico de acciones para simular el comportamiento del usuario. Lo más importante es que **todas incluyen auto-espera**: Playwright espera a que el elemento sea visible, habilitado e interactuable antes de ejecutar la acción.

| Acción | Descripción | Ejemplo |
|--------|-------------|------|
| `click()` | Clic normal | `page.locator("button").click()` |
| `fill(text)` | Limpia el campo y escribe | `page.locator("input").fill("texto")` |
| `type(text)` | Escribe caracter por caracter (simula teclado) | `page.locator("input").type("texto")` |
| `press(key)` | Presiona tecla | `page.locator("input").press("Enter")` |
| `select_option(value)` | Selecciona en `<select>` | `page.locator("select").select_option("opcion")` |
| `hover()` | Pasa el cursor por encima | `page.locator(".menu").hover()` |
| `check()` / `uncheck()` | Checkbox | `page.locator("input[type=checkbox]").check()` |
| `screenshot()` | Captura de pantalla | `page.screenshot(path="fallo.png")` |
| `wait_for_selector()` | Espera explícita a un elemento | `page.wait_for_selector(".resultado")` |
| `wait_for_load_state()` | Espera estado de carga | `page.wait_for_load_state("networkidle")` |

In [ ]:
# ──────────────────────────────────────────────────────────────
# DEMOSTRACIÓN DE ACCIONES — Ciclo completo: crear y completar
# ──────────────────────────────────────────────────────────────
from playwright.sync_api import sync_playwright

with sync_playwright() as pw:
    browser = pw.chromium.launch(headless=True)
    page = browser.new_page()
    page.goto(BASE_URL)

    # Limpiar estado inicial
    import urllib.request
    urllib.request.urlopen(
        urllib.request.Request(f"{BASE_URL}/tasks/clear", method="POST")
    )
    page.reload()

    print("--- Acción: fill + click para crear tarea ---")
    page.fill("[data-testid='input-titulo']", "Aprender Playwright")
    page.click("[data-testid='btn-agregar']")
    page.wait_for_load_state("networkidle")

    print("--- Verificando que la tarea aparece ---")
    tareas = page.locator("[data-testid='tarea-titulo']")
    print(f"  Número de tareas visibles: {tareas.count()}")
    print(f"  Título de la primera tarea: '{tareas.first.inner_text()}'")

    print("--- Acción: completar la tarea ---")
    page.click("[data-testid='btn-completar']")
    page.wait_for_load_state("networkidle")

    print("--- Verificando badge de completada ---")
    badge = page.locator("[data-testid='badge-completada']")
    print(f"  Badge 'Completada' visible: {badge.is_visible()}")
    print(f"  Texto del badge: '{badge.inner_text()}'")

    # Screenshot del estado final
    page.screenshot(path="screenshot_completada.png")
    print("  Screenshot guardado: screenshot_completada.png")

    browser.close()

---

## 6. El sistema `expect()` — Aserciones inteligentes

El método `expect()` de Playwright es mucho más poderoso que un simple `assert`. Incluye:

1. **Reintento automático:** Playwright sigue evaluando la condición durante un tiempo configurable (por defecto 5 segundos) antes de declarar el fallo. Esto elimina los fallos por timing.
2. **Mensajes de error descriptivos:** Indica exactamente qué esperaba y qué encontró.
3. **Matchers específicos para UI:** `to_be_visible()`, `to_contain_text()`, `to_have_count()`, etc.

### Comparación: `assert` simple vs `expect()`

```python
# ❌ FRÁGIL: falla si el elemento no está listo todavía
titulo = page.locator("[data-testid='tarea-titulo']").first
assert titulo.inner_text() == "Mi tarea"  # puede fallar por timing

# ✅ ROBUSTO: espera hasta que la condición sea verdadera
from playwright.sync_api import expect
expect(page.locator("[data-testid='tarea-titulo']").first).to_have_text("Mi tarea")
```

### Matchers más usados

In [ ]:
from playwright.sync_api import sync_playwright, expect

with sync_playwright() as pw:
    browser = pw.chromium.launch(headless=True)
    page = browser.new_page()

    # Limpiar y preparar estado
    import urllib.request
    urllib.request.urlopen(
        urllib.request.Request(f"{BASE_URL}/tasks/clear", method="POST")
    )
    page.goto(BASE_URL)

    print("=== Demostración de expect() matchers ===")

    # 1. to_be_visible() / to_be_hidden()
    expect(page.locator("[data-testid='form-nueva-tarea']")).to_be_visible()
    print("✓ to_be_visible(): el formulario está visible")

    # 2. to_have_text() — verifica el texto exacto
    expect(page.locator("[data-testid='page-title']")).to_have_text("📋 Gestor de Tareas")
    print("✓ to_have_text(): el título tiene el texto correcto")

    # 3. to_be_empty() — verifica que el input está vacío
    expect(page.locator("[data-testid='input-titulo']")).to_be_empty()
    print("✓ to_be_empty(): el input inicia vacío")

    # 4. to_have_count() — número de elementos
    expect(page.locator("[data-testid='tarea-item']")).to_have_count(0)
    print("✓ to_have_count(0): no hay tareas inicialmente")

    # Creamos una tarea para más demostraciones
    page.fill("[data-testid='input-titulo']", "Tarea de ejemplo")
    page.click("[data-testid='btn-agregar']")

    # 5. to_have_count() después de crear
    expect(page.locator("[data-testid='tarea-item']")).to_have_count(1)
    print("✓ to_have_count(1): hay exactamente 1 tarea después de crearla")

    # 6. to_contain_text() — verifica substring
    expect(page.locator("[data-testid='tarea-titulo']").first).to_contain_text("ejemplo")
    print("✓ to_contain_text(): el título contiene 'ejemplo'")

    # 7. not_to_be_visible() — verificar ausencia
    expect(page.locator("[data-testid='msg-lista-vacia']")).not_to_be_visible()
    print("✓ not_to_be_visible(): el mensaje 'vacío' no se muestra cuando hay tareas")

    browser.close()
    print("\nTodas las aserciones con expect() pasaron.")

---

## 7. Diagnóstico de fallos: Screenshots y Trazas

Una de las ventajas más prácticas de Playwright es su capacidad para generar artefactos de diagnóstico cuando un test falla.

### Screenshots
```python
# Captura toda la página
page.screenshot(path="fallo.png")

# Captura un elemento específico
page.locator("[data-testid='lista-tareas']").screenshot(path="lista.png")
```

### Trazas (Playwright Trace Viewer)
Una traza graba **todo** lo que ocurrió durante el test: screenshots de cada acción, logs de red, consola del browser, DOM en cada momento.

```python
context.tracing.start(screenshots=True, snapshots=True)
# ... ejecutar el test ...
context.tracing.stop(path="trace.zip")
# Ver la traza:
# playwright show-trace trace.zip
```

### En pytest con pytest-playwright
El plugin `pytest-playwright` automatiza los screenshots en fallos si se configura:
```bash
pytest --screenshot=on-failure --video=retain-on-failure
```

---

## 8. Ejecución headless vs headful

| Modo | Descripción | Cuándo usar |
|------|-------------|-------------|
| **headless=True** | Browser sin ventana visible (por defecto) | CI/CD, ejecución automática |
| **headless=False** | Browser con ventana visible | Desarrollo, diagnóstico visual |
| **slow_mo=500** | Ralentiza acciones (ms) | Depuración visual paso a paso |

```python
# Modo de desarrollo (visible y lento para ver qué pasa)
browser = pw.chromium.launch(headless=False, slow_mo=500)

# Modo CI (rápido e invisible)
browser = pw.chromium.launch(headless=True)
```

---

## 9. Ejercicio integrador

In [ ]:
# ──────────────────────────────────────────────────────────────
# EJERCICIO: Escribe una función que realice el flujo completo
# Crear → Completar → Eliminar usando Playwright y expect()
# ──────────────────────────────────────────────────────────────
from playwright.sync_api import sync_playwright, expect
import urllib.request

def flujo_completo_tarea(titulo: str):
    """
    Automatiza el flujo completo de una tarea:
    1. Crea la tarea con el título dado.
    2. Verifica que aparece en la lista.
    3. La completa y verifica el badge.
    4. La elimina y verifica que desaparece.
    """
    with sync_playwright() as pw:
        browser = pw.chromium.launch(headless=True)
        page = browser.new_page()

        # Limpiar estado
        urllib.request.urlopen(
            urllib.request.Request(f"{BASE_URL}/tasks/clear", method="POST")
        )
        page.goto(BASE_URL)

        # PASO 1: Crear
        page.fill("[data-testid='input-titulo']", titulo)
        page.click("[data-testid='btn-agregar']")
        expect(page.locator("[data-testid='tarea-titulo']").first).to_have_text(titulo)
        print(f"✓ Tarea '{titulo}' creada y visible")

        # PASO 2: Completar
        page.click("[data-testid='btn-completar']")
        expect(page.locator("[data-testid='badge-completada']")).to_be_visible()
        print(f"✓ Tarea '{titulo}' completada — badge visible")

        # PASO 3: Eliminar
        page.click("[data-testid='btn-eliminar']")
        expect(page.locator("[data-testid='tarea-item']")).to_have_count(0)
        expect(page.locator("[data-testid='msg-lista-vacia']")).to_be_visible()
        print(f"✓ Tarea '{titulo}' eliminada — lista vacía")

        browser.close()


flujo_completo_tarea("Dominar Playwright")
print("\nFlujo completo ejecutado con éxito.")

---

## 10. Preguntas para el informe

1. **Locators:** ¿Por qué `data-testid` es más robusto que un selector CSS como `.btn-complete`? ¿Qué cambio en el código de la aplicación rompería un locator CSS pero no un `data-testid`?

2. **Auto-esperas:** ¿Qué problema resuelven las auto-esperas de Playwright? Da un ejemplo de una situación donde sin auto-esperas el test fallaría intermitentemente.

3. **expect() vs assert:** ¿Cuándo usarías `expect(locator).to_have_text("X")` en lugar de `assert locator.inner_text() == "X"`? ¿Qué ventaja concreta tiene el primero?

4. **Trazas:** Si un test E2E falla solo en el servidor de CI pero no en tu máquina local, ¿cómo usarías las trazas de Playwright para diagnosticar el problema?

5. **Ejercicio extra:** Modifica la función `flujo_completo_tarea()` para que tome un screenshot en cada paso y los guarde con nombres descriptivos. ¿Cómo organizarías estos screenshots en un proyecto real?